# McDonald's diet problem: LP solution cases

The earlier diet notebooks minimize the cost of meeting nutrient minimums. Here we change the objective and add serving limits to explore three possible LP outcomes: **unbounded**, **optimal**, and **infeasible**. We then relax one constraint to restore feasibility.

Open the course repository root in VS Code, select the Julia 1.12 kernel with the course environment, and choose **Run All**. JuMP, HiGHS, NamedArrays, and Printf are already included. No external data files are needed.

The unbounded and infeasible cases are intentional. Their cells report the solver status and continue without trying to print an optimal menu. All models allow fractional servings and use the original classroom data.

## Menu and nutrient data

We reuse the data from `03-McDonaldsDiet.ipynb`. Every nutrient constraint is a lower bound; there are no upper limits on nutrient intake.

| Symbol | Menu item |
| --- | --- |
| `:QP` | Quarter Pounder |
| `:MD` | McLean Deluxe |
| `:BM` | Big Mac |
| `:FF` | Filet-O-Fish |
| `:MC` | McGrilled Chicken |
| `:FR` | Small Fries |
| `:SM` | Sausage McMuffin |
| `:M1` | 1% Milk |
| `:OJ` | Orange Juice |

The rows of `A` are protein, vitamin A, vitamin C, calcium, iron, calories, and carbohydrates. Each minimum uses the same units as its row of `A`. `A_NA[i, j]` selects a nutrient amount by its row and food labels.

In [ ]:
using JuMP, HiGHS, NamedArrays, Printf
import MathOptInterface as MOI

foods = [:QP, :MD, :BM, :FF, :MC, :FR, :SM, :M1, :OJ]
nutrients = [:Prot, :VitA, :VitC, :Calc, :Iron, :Cals, :Carb]

cost = Dict(zip(foods, [1.84, 2.19, 1.84, 1.44, 2.29, 0.77, 1.29, 0.6, 0.72]))
required = Dict(zip(nutrients, [55, 100, 100, 100, 100, 2000, 350]))

# Rows are nutrients; columns are foods, in the orders given above.
A = [
    28 24 25 14 31 3 15 9 1
    15 15 6 2 8 0 4 10 2
    6 10 2 0 15 15 0 4 120
    30 20 25 15 15 0 20 30 2
    20 20 20 10 8 2 15 0 2
    510 370 500 370 400 220 345 110 80
    34 33 42 38 42 26 27 12 20
]
A_NA = NamedArray(A, (nutrients, foods), ("Nutrients", "Menu Items"))
A_NA

## 1. Maximize hamburgers: an unbounded LP

Suppose I want to maximize the number of hamburgers I eat. I like hamburgers a lot! Let $B = \{\mathrm{QP}, \mathrm{MD}, \mathrm{BM}\}$. Replace the cost objective with

$$
\max \sum_{j \in B} x_j.
$$

Keep the nutrient minimums and nonnegative servings. Before running the model, consider whether anything limits how many hamburgers we can eat.

In [ ]:
burgers = [:QP, :MD, :BM]

unbounded_model = Model(HiGHS.Optimizer)
set_silent(unbounded_model)  # Remove this line to see the solver log.

@variable(unbounded_model, x_unbounded[foods] >= 0)
@objective(unbounded_model, Max, sum(x_unbounded[j] for j in burgers))
@constraint(unbounded_model, nutrient_minimum[i in nutrients],
    sum(A_NA[i, j] * x_unbounded[j] for j in foods) >= required[i])

optimize!(unbounded_model)
unbounded_status = termination_status(unbounded_model)
println("Termination status: ", unbounded_status)
println("Primal status: ", primal_status(unbounded_model))
println("Solver detail: ", raw_status(unbounded_model))

if unbounded_status == MOI.DUAL_INFEASIBLE
    println("This LP is feasible and unbounded; it has no optimal menu.")
elseif unbounded_status == MOI.INFEASIBLE_OR_UNBOUNDED
    println("The solver did not distinguish infeasibility from unboundedness.")
    println("The argument below shows that this LP is feasible and unbounded.")
else
    error("Unexpected status for the unbounded example: $(unbounded_status).")
end

### Why there is no optimal menu

Ten servings of every food satisfy all nutrient minimums. Starting with that feasible menu, we can add as many Quarter Pounders as we like. All nutrient coefficients are nonnegative, so adding burgers preserves feasibility and increases the objective without limit.

`DUAL_INFEASIBLE` alone does not prove that an arbitrary LP is unbounded: we also need to know that the primal LP is feasible. The argument above establishes both feasibility and unboundedness here. We will study duality later.

Check `termination_status` and the availability of a feasible solution before interpreting objective or variable values. In an unbounded solve, returned values may describe an improving direction (an **unbounded ray**) instead of an optimal menu. See the [JuMP solutions manual](https://jump.dev/JuMP.jl/stable/manual/solutions/).

For the optimal cases below, we require `MOI.OPTIMAL` and `is_solved_and_feasible(model)` before reporting values.

## 2. Limit each food: an optimal LP

Now impose $0 \leq x_j \leq 10$ for every food. We still maximize hamburgers.

The three burger variables can contribute at most $10 + 10 + 10 = 30$ servings. Ten servings of every food are feasible, so the maximum must be 30. The amounts of other foods may vary between optimal menus because they do not appear in the objective.

In [ ]:
max_item = Dict(j => 10 for j in foods)

bounded_model = Model(HiGHS.Optimizer)
set_silent(bounded_model)

@variable(bounded_model, 0 <= x_bounded[j in foods] <= max_item[j])
@objective(bounded_model, Max, sum(x_bounded[j] for j in burgers))
@constraint(bounded_model, nutrient_minimum[i in nutrients],
    sum(A_NA[i, j] * x_bounded[j] for j in foods) >= required[i])

optimize!(bounded_model)
bounded_status = termination_status(bounded_model)
println("Termination status: ", bounded_status)
bounded_status == MOI.OPTIMAL || error("HiGHS stopped with status $(bounded_status).")
is_solved_and_feasible(bounded_model) || error("No feasible optimal solution is available.")

maximum_burgers = objective_value(bounded_model)
bounded_solution = Dict(j => value(x_bounded[j]) for j in foods if value(x_bounded[j]) > 1e-6)

@printf("\nMaximum hamburger servings: %.2f\n", maximum_burgers)
for j in foods
    if haskey(bounded_solution, j)
        @printf("Eat %.2f servings of menu item %s\n", bounded_solution[j], j)
    end
end

## 3. Add daily serving limits: an infeasible LP

Return to minimizing cost and build a new model with these limits:

1. At most 3 sandwiches per day.
2. At most 3 drinks per day.
3. At most 2 orders of fries per day.

The sandwiches are `:QP`, `:MD`, `:BM`, `:FF`, `:MC`, and `:SM`; drinks are `:M1` and `:OJ`. Fries are `:FR` (`:FF` is Filet-O-Fish). These category limits replace the previous example's bound of 10 on each food.

Can a menu meet all nutrient minimums under these limits?

In [ ]:
sandwiches = [:QP, :MD, :BM, :FF, :MC, :SM]

limited_model = Model(HiGHS.Optimizer)
set_silent(limited_model)

@variable(limited_model, x_limited[foods] >= 0)
@objective(limited_model, Min, sum(cost[j] * x_limited[j] for j in foods))
@constraint(limited_model, nutrient_minimum[i in nutrients],
    sum(A_NA[i, j] * x_limited[j] for j in foods) >= required[i])

# Named constraints give us references to inspect or delete later.
@constraint(limited_model, MaxSandwiches, sum(x_limited[j] for j in sandwiches) <= 3)
@constraint(limited_model, MaxDrinks, x_limited[:M1] + x_limited[:OJ] <= 3)
@constraint(limited_model, MaxFries, x_limited[:FR] <= 2)

optimize!(limited_model)
infeasible_status = termination_status(limited_model)
println("Termination status: ", infeasible_status)
println("Primal status: ", primal_status(limited_model))

if infeasible_status == MOI.INFEASIBLE
    println("No menu satisfies all nutrient minimums and daily serving limits.")
else
    error("Unexpected status for the infeasible example: $(infeasible_status).")
end

### Explain the infeasibility

Each sandwich supplies at most 15 units of vitamin A, each drink at most 10, and fries supply none. The category limits therefore allow at most

$$
3(15) + 3(10) + 2(0) = 75
$$

units of vitamin A, below the required 100. No choice of objective can make these constraints feasible.

There is no feasible menu to report. In particular, we should not call `value` or `objective_value` to interpret this solve as a diet.

## 4. Relax the drink limit

Suppose we suspect that allowing more drinks can restore feasibility. Introduce $s \geq 0$, the number of drinks allowed beyond three, and replace the drink constraint with

$$
x_{\mathrm{M1}} + x_{\mathrm{OJ}} - s \leq 3.
$$

Change the objective to $\min s$. This finds the **smallest increase in the drink allowance** that makes all remaining constraints feasible. It does not minimize cost.

We modify the same model: delete the original drink constraint, add a variable and a replacement constraint, and overwrite the objective. The sandwich, fries, and nutrient constraints remain in place. After modifying the model, solve it again before reading any solution values.

In [ ]:
delete(limited_model, MaxDrinks)
@variable(limited_model, s >= 0)
@constraint(limited_model, RelaxedMaxDrinks, x_limited[:M1] + x_limited[:OJ] - s <= 3)
@objective(limited_model, Min, s)

optimize!(limited_model)
relaxed_status = termination_status(limited_model)
println("Termination status: ", relaxed_status)
relaxed_status == MOI.OPTIMAL || error("HiGHS stopped with status $(relaxed_status).")
is_solved_and_feasible(limited_model) || error("No feasible optimal solution is available.")

extra_drinks = objective_value(limited_model)
total_drinks = value(x_limited[:M1]) + value(x_limited[:OJ])
relaxed_solution = Dict(j => value(x_limited[j]) for j in foods if value(x_limited[j]) > 1e-6)

@printf("\nMinimum extra drinks beyond three: %.2f\n", extra_drinks)
@printf("Total drinks: %.2f (%.2f milk, %.2f orange juice)\n",
    total_drinks, value(x_limited[:M1]), value(x_limited[:OJ]))
for j in foods
    if haskey(relaxed_solution, j)
        @printf("Eat %.2f servings of menu item %s\n", relaxed_solution[j], j)
    end
end

## Interpret the cases

| Model | Outcome | Meaning |
| --- | --- | --- |
| Maximize hamburgers with only nutrient minimums | Unbounded | Feasible menus can have arbitrarily many hamburgers. |
| Add a limit of 10 servings per food | Optimal | The maximum is 30 hamburger servings. |
| Minimize cost with the sandwich, drink, and fries limits | Infeasible | No menu meets all constraints. |
| Minimize extra drinks while keeping the other limits | Optimal | Increasing the drink allowance restores feasibility. |

Which constraints are binding in the relaxed solution? Does the vitamin A argument alone determine the number of extra drinks needed, or do other nutrients force a larger increase?

The printed servings are rounded; the model uses unrounded values. Rounding a menu can violate its constraints. To rerun the whole example, choose **Run All**, which rebuilds each model before modifying the drink limit.